# Inter-Agent Communication (A2A)

The Agent-to-Agent (A2A) pattern enables independent AI agents to communicate, delegate tasks, and collaborate. A2A defines a standard: Agent Cards for discovery, typed Messages for communication, and HTTP/JSON-RPC for transport.

## Implementation with Flyte v2 + the Agent harness

This notebook reimplements the A2A calendar pattern using **Flyte v2 `Agent`s composed as tools**. Each specialist (calendar, scheduler, notifier) is its own `Agent` wrapped in an `@env.task`; a **planner `Agent`** holds those tasks as tools and delegates to them. The hand-written `asyncio.gather` orchestration becomes the planner's own tool-calling loop — and every delegated call shows up as a nested sub-agent run in the Flyte UI.

#### A2A / ADK vs Flyte v2 + Agent harness

| Aspect | A2A / ADK | Flyte v2 + `Agent` harness |
|--------|-----------|----------------------------|
| **Agent Card** | JSON at `/.well-known/agent.json` | `TaskEnvironment(name=..., description=...)` |
| **Discovery** | Well-known URI / registry | `flyte.remote.Task.get("team.agent_name")` |
| **Delegation** | `sendTask` over HTTP/JSON-RPC | Planner `Agent` calls specialist `@env.task` tools |
| **Orchestration** | Client code wiring calls | The planner's tool-calling loop |
| **Message format** | `Message(role, parts=[...])` | Plain strings between agents |
| **Auth** | OAuth 2.0 / API key headers | `flyte.Secret` + cluster RBAC |
| **Observability** | SSE / external tracing | Nested sub-agent runs in the Flyte UI |

> **🧭 When to use this pattern — and how Flyte helps**
>
> Use agent-to-agent communication when two or more autonomous agents — possibly from different teams or frameworks — must coordinate, delegate, and combine results. In Flyte each specialist is a separately deployable task discoverable via `flyte.remote.Task.get`, so a planner agent delegates to them as tools through durable, typed hand-offs, no standing HTTP services required.

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' litellm

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure TaskEnvironments

Each `TaskEnvironment` plays the role of an **Agent Card** in A2A — it declares the agent's name, capabilities (resources, image), and authentication requirements (secrets). In A2A, separate agents might run as distinct services; in Flyte v2, they run as separate task environments on the same cluster.

In [5]:

import os
import uuid
from datetime import datetime, timedelta

import flyte
from flyte.ai.agents import Agent, AgentResult

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="a2a-agents", python_version=(3, 12))
    .with_pip_packages("litellm")
)

_secrets = [flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")]

# ── Agent Card equivalents (one TaskEnvironment per specialist) ───────────────

calendar_env = flyte.TaskEnvironment(
    name="calendar_agent", image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"), secrets=_secrets,
)
scheduler_env = flyte.TaskEnvironment(
    name="scheduler_agent", image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"), secrets=_secrets,
)
notifier_env = flyte.TaskEnvironment(
    name="notifier_agent", image=_image,
    resources=flyte.Resources(cpu="1", memory="512Mi"), secrets=_secrets,
)
planner_env = flyte.TaskEnvironment(
    name="planner_agent", image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"), secrets=_secrets,
    depends_on=[calendar_env, scheduler_env, notifier_env]
)

### 4. Define the specialist agents (as tools)

Each specialist is its own `Agent` with a focused instruction set. Wrapping each one in an `@env.task` turns it into a **tool** the planner can call: the planner asks for availability, a slot, or a notification, and the corresponding sub-agent runs its own LLM loop inside its own task (and its own container/Agent Card). This is the "agents as tools" composition.   

Each specialist agent (`calendar_env`, `scheduler_env`, `notifier_env`) can be deployed independently and called via `flyte.remote.Task.get()` by any other team — the Flyte equivalent of A2A's agent registry and `sendTask`. Because each specialist is wrapped as an `@env.task`, the planner's delegations appear as nested sub-agent runs in the UI, no HTTP plumbing required.

In [6]:
# ── Calendar specialist ───────────────────────────────────────────────────────
calendar_agent = Agent(
    name="calendar",
    model="claude-haiku-4-5",
    instructions=(
        "You are a calendar assistant in a scheduling SIMULATION. You have no real "
        "calendar access, so you INVENT plausible availability. Given a person and a date "
        "range, ALWAYS output 2-3 realistic time slots that fall within that range, one per "
        "line as 'DATE | HH:MM-HH:MM | available/busy' (mostly 'available', occasionally "
        "'busy'). Output ONLY the slots — never refuse, never ask for more information, "
        "never mention lacking calendar access, and ignore whether the dates are past or future."
    ),
)


@calendar_env.task(retries=2, timeout=timedelta(minutes=2), cache="auto")
async def check_availability(user_name: str, date_range: str, duration_minutes: int = 60) -> str:
    """Check one person's calendar availability for a date range.

    Args:
        user_name: Whose calendar to check.
        date_range: e.g. '2025-08-11 to 2025-08-15'.
        duration_minutes: Required meeting length.
    """
    r = await calendar_agent.run.aio(
        f"Invent availability for {user_name}. Date range: {date_range}. "
        f"Duration: {duration_minutes} minutes."
    )
    return f"{user_name}:\n{r.summary}"


# ── Scheduler specialist ──────────────────────────────────────────────────────
scheduler_agent = Agent(
    name="scheduler",
    model="claude-haiku-4-5",
    instructions=(
        "You are a scheduling assistant. Given several people's availability, pick ONE "
        "slot that works for everyone and reply with just 'DATE | HH:MM-HH:MM'. If nothing "
        "overlaps, choose the organizer's first available slot. Reply with only the slot."
    ),
)


@scheduler_env.task(retries=1, timeout=timedelta(minutes=2))
async def pick_slot(availabilities: str) -> str:
    """Choose one common meeting slot from several people's availability text.

    Args:
        availabilities: Concatenated availability listings for all attendees.
    """
    r = await scheduler_agent.run.aio(availabilities)
    return r.summary


# ── Notifier specialist ───────────────────────────────────────────────────────
notifier_agent = Agent(
    name="notifier",
    model="claude-haiku-4-5",
    instructions="You are a notifier. Write a concise 3-sentence meeting confirmation message.",
)


@notifier_env.task(retries=1, timeout=timedelta(minutes=1))
async def notify_attendees(meeting_title: str, attendees: str, slot: str) -> str:
    """Compose and send a meeting confirmation to the attendees.

    Args:
        meeting_title: The meeting subject.
        attendees: Comma-separated attendee names.
        slot: The chosen slot, e.g. '2025-08-12 | 10:00-11:00'.
    """
    confirmation_id = f"MTG-{uuid.uuid4().hex[:8].upper()}"
    r = await notifier_agent.run.aio(
        f"Confirm '{meeting_title}' for {attendees} at {slot}. Confirmation: {confirmation_id}."
    )
    return f"[{confirmation_id}] sent to {attendees}: {r.summary}"

### 5. Define the orchestrating planner agent

The planner `Agent`  **plans the delegation itself** — it decides to call `check_availability` per person, then `pick_slot`, then `notify_attendees`, calling each specialist sub-agent as a tool. To call a specialist owned by another team, swap the local task for `flyte.remote.Task.get("platform_team.check_availability", auto_version="latest")`.

In [7]:
planner_agent = Agent(
    name="planner",
    model="claude-sonnet-4-6",
    instructions=(
        "You schedule meetings by delegating to specialist agents. Steps:\n"
        "1. Call check_availability for the organizer and for each attendee.\n"
        "2. Call pick_slot with all the availability results to choose one common slot.\n"
        "3. Call notify_attendees with the meeting title, attendees, and chosen slot.\n"
        "Finish with a one-line confirmation of what was scheduled."
    ),
    tools=[check_availability, pick_slot, notify_attendees],
    max_turns=12,
)


@planner_env.task(
    retries=1,
    timeout=timedelta(minutes=8),
    cache=flyte.Cache(behavior="disable"),
)
async def schedule_meeting(
    organizer: str,
    attendees: list[str],
    meeting_title: str,
    preferred_dates: str,
    duration_minutes: int = 60,
) -> str:
    """Planner agent: orchestrates the calendar, scheduler, and notifier sub-agents.

    Each specialist runs as its own nested task/sub-agent in the Flyte UI — the A2A
    delegation pattern without an HTTP server or JSON-RPC envelope.
    """
    request = (
        f"Schedule '{meeting_title}'. Organizer: {organizer}. "
        f"Attendees: {', '.join(attendees)}. Preferred dates: {preferred_dates}. "
        f"Duration: {duration_minutes} minutes."
    )
    result: AgentResult = await planner_agent.run.aio(request)
    if result.error:
        raise RuntimeError(result.error)
    return result.summary

### 6. Run on the devbox

In [8]:
from datetime import date, timedelta

# Future-relative dates so the demo never asks the agents to schedule in the past.
_start = date.today() + timedelta(days=7)
_end = _start + timedelta(days=4)
preferred_dates = f"{_start:%Y-%m-%d} to {_end:%Y-%m-%d}"

run = flyte.run(
    schedule_meeting,
    organizer="Alice",
    attendees=["Bob", "Carol"],
    meeting_title="Q3 Planning Session",
    preferred_dates=preferred_dates,
    duration_minutes=90,
)
run.wait()
print(run.outputs()[0])
print(run.url)

> Building 4 images...

> Building image a2a-agents for environment planner_agent

> Building image a2a-agents for environment calendar_agent

> Building image a2a-agents for environment scheduler_agent

> Building image a2a-agents for environment notifier_agent

...

✓ Built image for environment calendar_agent: localhost:30000/a2a-agents:c0e0f4726ed0c807007057a3d5d61d37

✓ Built image for environment scheduler_agent: localhost:30000/a2a-agents:c0e0f4726ed0c807007057a3d5d61d37

✓ Built image for environment notifier_agent: localhost:30000/a2a-agents:c0e0f4726ed0c807007057a3d5d61d37

Output()

✅ **"Q3 Planning Session"** has been successfully scheduled for **June 30, 2026, from 2:00 PM – 3:30 PM**, with Alice (organizer), Bob, and Carol — confirmation #MTG-4063CDAA sent to all attendees.
http://localhost:30080/v2/domain/development/project/flytesnacks/runs/r4sggdt28958brxb2qj6
